<a href="https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/integration/evaluation_with_deepeval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>   <a href="https://github.com/milvus-io/bootcamp/blob/master/integration/evaluation_with_deepeval.ipynb" target="_blank">
    <img src="https://img.shields.io/badge/View%20on%20GitHub-555555?style=flat&logo=github&logoColor=white" alt="GitHub Repository"/>
</a>


# Evaluation with DeepEval

This guide demonstrates how to use [DeepEval](https://docs.confident-ai.com/) to evaluate a Retrieval-Augmented Generation (RAG) pipeline built upon [Milvus](https://milvus.io/).

The RAG system combines a retrieval system with a generative model to generate new text based on a given prompt. The system first retrieves relevant documents from a corpus using Milvus, and then uses a generative model to generate new text based on the retrieved documents.

DeepEval is a framework that helps you evaluate your RAG pipelines. There are existing tools and frameworks that help you build these pipelines but evaluating it and quantifying your pipeline performance can be hard. This is where DeepEval comes in.

In [2]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path[0]}")

Project root: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph
Python path: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph


We will use OpenAI as the LLM in this example. You should prepare the [api key](https://platform.openai.com/docs/quickstart) `OPENAI_API_KEY` as an environment variable.

In [3]:
# Load environment variables
from dotenv import load_dotenv

env_file = project_root / ".env"
if env_file.exists():
    load_dotenv(env_file, override=True)
    print("✓ Environment variables loaded")
else:
    print("⚠ No .env file found")

# Verify required environment variables
required_vars = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "GOOGLE_API_KEY", "CHAT_OPENAI_API_KEY"]
missing = [var for var in required_vars if not os.getenv(var)]

if missing:
    print(f"❌ Missing environment variables: {missing}")
else:
    print(f"✓ All required environment variables present")
    print(f"  - Neo4j URI: {os.getenv('NEO4J_URI')}")
    print(f"  - Neo4j Database: {os.getenv('NEO4J_DATABASE', 'neo4j')}")
    print(f"  - Embedding Model: {os.getenv('EMBEDDING_MODEL', 'gemini-embedding-001')}")

✓ Environment variables loaded
✓ All required environment variables present
  - Neo4j URI: bolt://localhost:7687
  - Neo4j Database: vn301
  - Embedding Model: gemini-embedding-001


## Define the RAG pipeline

We will define the RAG class that use Milvus as the vector store, and OpenAI as the LLM.
The class contains the `load` method, which loads the text data into Milvus, the `retrieve` method, which retrieves the most similar text data to the given question, and the `answer` method, which answers the given question with the retrieved knowledge.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
# from rag import build_graph, get_retriever
from rag.retrievers import get_retriever, build_graph
from typing import List
import os


class MultilayerGraphRAG:
    """
    Multilayer Graph RAG (Retrieval-Augmented Generation) class built upon Neo4j SPG Knowledge Graph.
    
    This wraps the SPG-based RAG system using vector similarity search for evaluation with DeepEval.
    Uses vector similarity on entity embeddings with full relationship properties.
    """

    def __init__(self):
        """Initialize the Multilayer Graph RAG system with Neo4j backend."""
        print("[MultilayerGraphRAG] Initializing Neo4j-based SPG Graph RAG...")
        
        # Initialize the graph and retriever
        self.graph = build_graph()
        self.retriever = get_retriever()
        
        # Initialize LLM with same config as agent
        self._init_llm()
        
        print("[MultilayerGraphRAG] Multilayer Graph RAG initialized successfully")
        print("[MultilayerGraphRAG] Using vector similarity search on SPG entities")

    def _init_llm(self):
        """Initialize LLM with same configuration as the agent."""
        base_url = os.getenv("CHAT_OPENAI_BASE_URL")
        api_key = os.getenv("CHAT_OPENAI_API_KEY")

        # If using custom base_url without api_key, use "EMPTY" as recommended by LangChain
        if base_url and not api_key:
            api_key = "EMPTY"

        llm_kwargs = {
            "model": os.getenv("CHAT_LLM_MODEL", "gpt-4o-mini"),
            "temperature": float(os.getenv("LLM_TEMPERATURE", "0.1")),
            "api_key": api_key,
        }

        # Add base_url if provided
        if base_url:
            llm_kwargs["base_url"] = base_url

            # Add headers to help bypass Cloudflare protection
            llm_kwargs["default_headers"] = {
                "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
                "Accept": "application/json",
                "Accept-Language": "en-US,en;q=0.9"
            }

        self.llm = ChatOpenAI(**llm_kwargs)
        print(f"[MultilayerGraphRAG] LLM initialized: {llm_kwargs['model']}")

    def retrieve(self, question: str, top_k: int = 5) -> List[str]:
        """
        Retrieve relevant information using vector similarity search on entity embeddings.
        
        This method uses semantic vector search on entities in the knowledge graph,
        then retrieves their relationships with full SPG property values and source chunks.
        
        Args:
            question: Query question
            top_k: Maximum number of context items to retrieve
            
        Returns:
            List of retrieved context strings with entities, relationships, and properties
        """
        print(f"\n[MultilayerGraphRAG.retrieve] Vector search for: {question[:60]}...")
        
        # Use vector similarity search method from retriever
        result = self.retriever.vector_similarity_search(
            query=question,
            limit=top_k,
            similarity_threshold=0.6
        )
        
        if not result or "No similar entities found" in result:
            return [f"No relevant information found in knowledge graph for: {question}"]
        
        # Split the formatted result into individual context items
        # Each entity starts with a markdown heading like "**EntityName**"
        contexts = []
        current_context = []
        
        for line in result.split('\n'):
            # Check if this is a new entity (starts with ** and has entity name)
            if line.strip().startswith('**') and '**' in line[2:] and current_context:
                # Save the previous context
                contexts.append('\n'.join(current_context))
                current_context = [line]
            else:
                current_context.append(line)
        
        # Add the last context
        if current_context:
            contexts.append('\n'.join(current_context))
        
        # Clean up empty contexts
        contexts = [ctx.strip() for ctx in contexts if ctx.strip()]
        
        if not contexts:
            return [result]  # Return as single context if splitting failed
        
        print(f"[MultilayerGraphRAG.retrieve] Returning {len(contexts)} entity contexts with relationships\n")
        return contexts[:top_k]

    def load(self, texts: List[str]):
        """
        Load text data into Neo4j knowledge graph.
        
        Note: This assumes you have already processed and loaded your data into Neo4j
        using the knowledge graph pipeline. This method is kept for interface compatibility
        but doesn't perform the actual loading.
        
        Args:
            texts: List of text documents (not used as data is pre-loaded in Neo4j)
        """
        print(f"[MultilayerGraphRAG] Note: Multilayer Graph RAG uses pre-loaded Neo4j SPG knowledge graph.")
        print(f"[MultilayerGraphRAG] Skipping load of {len(texts)} texts - ensure your Neo4j database is populated.")
        print(f"[MultilayerGraphRAG] Use the SPG pipeline to process and load documents into Neo4j.")

    def answer(
        self,
        question: str,
        retrieval_top_k: int = 2,
        return_retrieved_text: bool = False,
    ):
        """
        Answer a question using direct LLM invocation with retrieved context.

        This method:
        1. Retrieves relevant contexts using vector similarity search
        2. Directly invokes the LLM with the question and contexts
        3. Returns the generated answer

        Args:
            question: Question to answer
            retrieval_top_k: Number of context items to retrieve
            return_retrieved_text: Whether to return retrieved contexts along with answer

        Returns:
            Answer string, or tuple of (answer, contexts) if return_retrieved_text=True
        """
        # Retrieve contexts using vector search
        print(f"\n[MultilayerGraphRAG] Processing question: {question[:80]}...")

        pre_process = f"""You are a financial analyst assistant. Your mission is to extract the key information needed to query the knowledge graph in order to find relevant data for answering the user's question.

Question: {question}

Example:
Question: Assume that you are a public equities analyst. Answer the following question by primarily using information shown in the balance sheet: what is the year-end FY2018 net PPNE for 3M? Answer in USD billions.

Answer: Consolidated Balance Sheet for FY2018 of 3M Company

Rule: 
1. Do not include your thinking, only the term of thing need to find.
2. Do not fabricate numbers or data; only return the names of entities or concepts that need to be found.
"""
        pre_process_question = self.llm.invoke([HumanMessage(content=pre_process)])

        print(pre_process_question.content)

        contexts = self.retrieve(pre_process_question.content, top_k=retrieval_top_k)

        # Format contexts for the prompt
        context_str = "\n\n".join([f"Context {i+1}:\n{ctx}" for i, ctx in enumerate(contexts)])

        print(context_str)

        # Create prompt with question and contexts
        prompt = f"""You are a financial analyst assistant. Answer the question using only the provided contexts from the knowledge graph.

Contexts:
{context_str}

Question: {question}

Instructions:
1. Provide a direct, concise answer that addresses what the user is asking
2. Focus on the key finding - avoid unnecessary background or explanations
3. If the contexts lack sufficient data to answer, state this clearly but returning which data is missing.
4. Make the answer short and concise

Answer:"""

        # Invoke LLM directly
        response = self.llm.invoke([HumanMessage(content=prompt)])
        answer = response.content

        print(f"[MultilayerGraphRAG] Generated answer using direct LLM invocation\n")

        if return_retrieved_text:
            return answer, contexts

        return answer

    def answer_with_graph_context(
        self,
        question: str,
        return_retrieved_text: bool = False,
    ):
        """
        Answer a question and extract the actual retrieved contexts from tool calls.
        
        This method inspects the agent's tool calls to get the actual retrieved contexts
        used during reasoning, providing more accurate evaluation data.
        
        Args:
            question: Question to answer
            return_retrieved_text: Whether to return retrieved contexts
            
        Returns:
            Answer string, or tuple of (answer, contexts) if return_retrieved_text=True
        """
        # Create initial state
        initial_state = {
            "messages": [HumanMessage(content=question)],
            "retrieved_context": "",
            "intent": "",
            "original_question": "",
            "entities": []
        }
        
        # Run the graph
        print(f"\n[MultilayerGraphRAG] Processing question: {question[:80]}...")
        result = self.graph.invoke(initial_state)
        
        # Extract answer and contexts from messages
        answer = None
        contexts = []
        
        for msg in result["messages"]:
            # Collect tool results (retrieved contexts)
            if isinstance(msg, ToolMessage):
                # Only include substantial tool results
                if len(msg.content) > 50 and "not found" not in msg.content.lower():
                    contexts.append(msg.content)
            
            # Get final answer
            elif isinstance(msg, AIMessage) and not msg.tool_calls:
                answer = msg.content
        
        if not answer:
            answer = "I couldn't generate an answer. Please try rephrasing your question."
        
        if not contexts:
            contexts = [f"No relevant context retrieved for: {question}"]
        
        if return_retrieved_text:
            return answer, contexts
        
        return answer

    def close(self):
        """Close the Neo4j connection."""
        if hasattr(self, 'retriever') and hasattr(self.retriever, 'driver'):
            self.retriever.driver.close()
            print("[MultilayerGraphRAG] Neo4j connection closed")

Let's initialize the RAG class with OpenAI and Milvus clients.

In [4]:
# Initialize MultilayerGraphRAG
# NOTE: Ensure your Neo4j database is running and populated with knowledge graph data
# The MultilayerGraphRAG system requires:
# 1. Neo4j database with populated SPG knowledge graph (entities, relationships, embeddings)
# 2. Environment variables set in .env file (NEO4J_URI, NEO4J_PASSWORD, GOOGLE_API_KEY, etc.)
# 3. OPENAI_API_KEY for the LLM agent

my_rag = MultilayerGraphRAG()

# Note: The load() method is no-op for MultilayerGraphRAG
# Data should be pre-loaded into Neo4j using the SPG knowledge graph pipeline
# Example:
# python run_server.py  # Start the server
# Use batch processing API endpoint to process documents into Neo4j
# See docs/BATCH_PROCESSING_GUIDE.md for details

[MultilayerGraphRAG] Initializing Neo4j-based SPG Graph RAG...

[Graph] Building RAG graph with ReAct agent...
[Graph] Flow: preprocess_query → agent (ReAct) → END
[Neo4j Retriever] Using Gemini embedding model: gemini-embedding-001
[Neo4j Retriever] Embedding dimension: 3072
[MultilayerGraphRAG] LLM initialized: openai/gpt-oss-20b
[MultilayerGraphRAG] Multilayer Graph RAG initialized successfully
[MultilayerGraphRAG] Using vector similarity search on SPG entities


Now let's prepare some questions with its corresponding ground truth answers. We get answers and contexts from our RAG pipeline.

In [9]:
# Import financebench dataset

import json
from datasets import Dataset
import pandas as pd
from tqdm import tqdm

data = []
with open("./.result/financebench_open_source.jsonl", 'r') as f:
    for line in f:
        line = line.strip()
        if line:  # Skip empty lines
            data.append(json.loads(line))
financebench = pd.DataFrame(data)


In [6]:
# Preprocessing context ground truth
financebench["evidence_text"] = financebench["evidence"].apply(
    lambda x: " ".join([item["evidence_text"] for item in x]) if isinstance(x, list) else ""
)

In [7]:
# Prepare data

question_list = []
ground_truth_list = []
contexts_list = []
retrieval_list = []
answer_list = []
for question, ground_truth, context in zip(financebench["question"], financebench["answer"], financebench["evidence_text"]):
    question_list.append(question)
    ground_truth_list.append(ground_truth)
    contexts_list.append(context)


In [8]:
print(f"Number of questions: {len(question_list)}")
print(f"Number of ground truths: {len(ground_truth_list)}")
print(f"Number of contexts: {len(contexts_list)}")

Number of questions: 150
Number of ground truths: 150
Number of contexts: 150


In [9]:
import time

retrieval_list = []
answer_list = []
for question in tqdm(question_list, desc="Answering questions"):
    answer, retrieval = my_rag.answer(question, return_retrieved_text=True)
    retrieval_list.append(retrieval)
    answer_list.append(answer)
    # time.sleep(3)

financebench_eval = pd.DataFrame(
    {
        "question": question_list,
        "contexts": contexts_list,
        "retrieval": retrieval_list,
        "answer": answer_list,
        "ground_truth": ground_truth_list,
    }
)
rag_results = Dataset.from_pandas(financebench_eval)
financebench_eval

Answering questions:   0%|          | 0/150 [00:00<?, ?it/s]


[MultilayerGraphRAG] Processing question: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a r...
Cash Flow Statement for FY2018 of 3M Company

[MultilayerGraphRAG.retrieve] Vector search for: Cash Flow Statement for FY2018 of 3M Company...
[MultilayerGraphRAG.retrieve] Returning 2 entity contexts with relationships

Context 1:
**Consolidated Statement of Cash Flows** (FinancialStatement) [similarity: 0.743]
  - ID: fs_mmm_cash_flows_2018
  - Description: A financial statement detailing the cash inflows and outflows of 3M Company for the fiscal year ended December 31, 2018, with comparative data for 2017 and 2016.
  - Source Chunks (Text Evidence):
    [Chunk 0, Page 59]: "Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$

Answering questions:   1%|          | 1/150 [00:07<18:25,  7.42s/it]

[MultilayerGraphRAG] Generated answer using direct LLM invocation


[MultilayerGraphRAG] Processing question: Assume that you are a public equities analyst. Answer the following question by ...
Net PPNE FY2018 3M

[MultilayerGraphRAG.retrieve] Vector search for: Net PPNE FY2018 3M...
[MultilayerGraphRAG.retrieve] Returning 2 entity contexts with relationships

Context 1:
**Consolidated Balance Sheet** (FinancialStatement) [similarity: 0.655]
  - ID: fs_mmm_balance_sheet_2018
  - Description: A financial statement detailing the assets, liabilities, and equity of 3M Company as of December 31, 2018, with comparative data for December 31, 2017.
  - Source Chunks (Text Evidence):
    [Chunk 0, Page 57]: "Table of Contents \n3M Company and Subsidiaries\nConsolidated Balance Shee t\nAt December 31\n \n \n \nDecember 31,\n \nDecember 31,\n \n(Dollars in millions, except per share amount)\n \n2018\n \n2017\n \nAssets\n \n \n \n \n \nCurrent assets\n \n \n \n \n \nCash and cash equivalents\n \n$

Answering questions:   1%|▏         | 2/150 [00:13<15:49,  6.42s/it]

...

[MultilayerGraphRAG] Generated answer using direct LLM invocation


[MultilayerGraphRAG] Processing question: How much has the effective tax rate of American Express changed between FY2021 a...
American Express effective tax rate FY2021  
American Express effective tax rate FY2022

[MultilayerGraphRAG.retrieve] Vector search for: American Express effective tax rate FY2021  
American Expres...
[MultilayerGraphRAG.retrieve] Returning 2 entity contexts with relationships

Context 1:
**Income Statement** (FinancialStatement) [similarity: 0.655]
  - ID: fs_american_express_statement_of_operations_2022
  - Description: A financial statement presenting American Express Company's revenues, expenses, and net income for the fiscal year ended December 31, 2022.
  - Source Chunks (Text Evidence):
    [Chunk 0, Page 43]: "Table of Contents\nTABLE 1: SUMMARY OF FINANCIAL PERFORMANCE\nYears Ended December 31,\nChange\nChange\n(Millions, except percentages, per share amounts and where indicated)\n2022

Answering questions:  29%|██▊       | 43/150 [05:41<16:36,  9.31s/it]

[MultilayerGraphRAG] Generated answer using direct LLM invocation


[MultilayerGraphRAG] Processing question: What was the largest liability in American Express's Balance Sheet in 2022?...


Answering questions:  29%|██▊       | 43/150 [06:48<16:55,  9.49s/it]


JSONDecodeError: Expecting value: line 315 column 1 (char 1727)

In [10]:
financebench_eval.to_csv('financebench_llama3-2-3B.csv', 
          index=False,           # Don't save row indices
          encoding='utf-8',      # UTF-8 encoding for special characters
          sep=',',
          quoting=1, 
          lineterminator='\n',
          escapechar='\\',
          ) 
print("Finished")

Finished


In [11]:
financebench_eval = pd.read_csv('./.result/financebench_gpt-oss-20B.csv',           # Don't save row indices
          encoding='utf-8',      # UTF-8 encoding for special characters
          quoting=1, 
          lineterminator='\n',
          escapechar='\\',
          )
financebench_eval.head()

,question,contexts,retrieval,answer,ground_truth
0,What is the FY2018 capital expenditure amount ...,Table of Contents \n3M Company and Subsidiarie...,"[""Error performing vector similarity search: F...",The provided context does not contain any fina...,$1577.00
1,Assume that you are a public equities analyst....,Table of Contents \n3M Company and Subsidiarie...,['**Consolidated Balance Sheet** (FinancialSta...,"$8.74 billion (Net Property, Plant & Equipment...",$8.70
2,Is 3M a capital-intensive business based on FY...,3M Company and Subsidiaries\nConsolidated Stat...,['**RESULTS OF OPERATIONS** (FinancialStatemen...,"**Answer:** \nBased on FY 2022 data, 3M is **...","No, the company is managing its CAPEX and Fixe..."
3,What drove operating margin change as of FY202...,"SG&A, measured as a percent of sales, increase...",['**RESULTS OF OPERATIONS** (FinancialStatemen...,**Operating‑margin change (FY 2022)** \n- 202...,Operating Margin for 3M in FY2022 has decrease...
4,"If we exclude the impact of M&A, which segment...",Worldwide Sales Change \nBy Business Segment\n...,['**Healthcare Industry** (Industry) [similari...,The provided contexts do not contain any finan...,The consumer segment shrunk by 0.9% organically.


In [5]:
financebench_eval = pd.read_csv('financebench_llama3-1-8B.csv',           # Don't save row indices
          encoding='utf-8',      # UTF-8 encoding for special characters
          quoting=1, 
          lineterminator='\n',
          escapechar='\\',
          )
financebench_eval.head()

NameError: name 'pd' is not defined

In [12]:
import ast
import pandas as pd

# If your DataFrame has a column with string representations of lists
financebench_eval['retrieval'] = financebench_eval['retrieval'].apply(ast.literal_eval)

type(financebench_eval['retrieval'][0])

list

## Evaluating Retriever

When evaluating a retriever in large language model (LLM) systems, it's crucial to assess the following:

1. **Ranking Relevance**: How effectively the retriever prioritizes relevant information over irrelevant data.
   
2. **Contextual Retrieval**: The ability to capture and retrieve contextually relevant information based on the input.

3. **Balance**: How well the retriever manages text chunk size and retrieval scope to minimize irrelevancies.

Together, these factors provide a comprehensive understanding of how the retriever prioritizes, captures, and presents the most useful information.

In [ ]:
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

contextual_precision = ContextualPrecisionMetric()
contextual_recall = ContextualRecallMetric()
contextual_relevancy = ContextualRelevancyMetric()

test_cases = []

for index, row in financebench_eval.iterrows():
    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        expected_output=row["ground_truth"],
        retrieval_context=row["retrieval"],
        context=row["contexts"]
    )
    test_cases.append(test_case)

# test_cases
result = evaluate(
    test_cases=test_cases,
    metrics=[contextual_precision, contextual_recall, contextual_relevancy],  # Change to True to see detailed metric results
)

## Evaluating Generation

To assess the quality of generated outputs in large language models (LLMs), it's important to focus on two key aspects:

1. **Relevance**: Evaluate whether the prompt effectively guides the LLM to generate helpful and contextually appropriate responses.
   
2. **Faithfulness**: Measure the accuracy of the output, ensuring the model produces information that is factually correct and free from hallucinations or contradictions. The generated content should align with the factual information provided in the retrieval context.

These factors together ensure that the outputs are both relevant and reliable.

In [21]:
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, HallucinationMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, ErrorConfig


# answer_relevancy = AnswerRelevancyMetric(model="gpt-5-mini")
faithfulness = FaithfulnessMetric(model="gpt-5-mini")
hallucination = HallucinationMetric(model="gpt-5-mini")
answer_relevancy = AnswerRelevancyMetric(model="gpt-5-mini")

test_cases = []

for index, row in financebench_eval.iterrows():
    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["answer"],
        # expected_output=row["ground_truth"],
        # retrieval_context=row["retrieval"],
        # context=row["retrieval"]
    )
    test_cases.append(test_case)

# test_cases
result = evaluate(
    test_cases=test_cases,
    metrics=[answer_relevancy],
    async_config=AsyncConfig(throttle_value=10),
    error_config=ErrorConfig(ignore_errors=True)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-5-mini, reason: The score is 1.00 because the response directly addressed the question using only balance-sheet-relevant information and stated 3M’s FY2018 net PP&E in USD billions with no irrelevant statements; it cannot be higher because 1.00 is the maximum possible score., error: None)

For test case:

  - input: Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
  - actual output: $8.74 billion (Net Property, Plant & Equipment at year‑end FY2018).
  - expected output: None
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-5-mini, reason: The score is 1.00 because the response contained no irrelevant statements and 

⚠ WARNING: No hyperparameters logged.
» ]8;id=341218;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

Sending a large test run to Confident, this might take a bit longer than usual...

✓ Done 🎉! View results on 
]8;id=517124;https://app.confident-ai.com/project/cmgaocias01i8ql0geax9kmf3/test-runs/cmjzpnioc2soppb1ey4411w58/regression-testing\https://app.confident-ai.com/project/cmgaocias01i8ql0geax9kmf3/test-runs/cmjzpnioc2soppb1ey4411w58/regression-testi]8;;\
]8;id=517124;https://app.confident-ai.com/project/cmgaocias01i8ql0geax9kmf3/test-runs/cmjzpnioc2soppb1ey4411w58/regression-testing\ng]8;;\